# 04 — Model training

Scaffold. Start from the reviewed feature and availability specification in
[03](03_feature_engineering.ipynb).

## Experiment setup

- Review the existing `scripts/train_qc_seasonal.py`,
  `scripts/backtest_rolling.py`, and `scripts/backtest_four_week.py` for reusable
  modeling and splitting logic. Check their calendar and input assumptions.
- Define chronological folds and reserve evaluation origins before tuning.
- Document candidate benchmarks, input sets, seeds, fit cutoffs, and experiment
  settings so the comparisons can be reproduced.

## Handoff

Provide dated or source-block-indexed predictions with origins, horizons, target
blocks, train cutoffs, and source hashes to [05 — Evaluation](05_evaluation.ipynb).

In [8]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

qc, case_column, _ = load_weekly("QUEZON_CITY")
cases = qc[case_column]
rain = qc["RF_NASA"]
origin = cases.index
results = []

for horizon in range(1, 5):
    target = origin + horizon
    season = 2 * np.pi * (target % 52) / 52
    rows = pd.DataFrame({
        "origin": origin,
        "target": target,
        "latest_cases": cases.to_numpy(),
        "last_year_cases": cases.reindex(target - 52).to_numpy(),
        "season_sin": np.sin(season),
        "season_cos": np.cos(season),
        "actual": cases.reindex(target).to_numpy(),
    })

    case_inputs = ["latest_cases", "last_year_cases", "season_sin", "season_cos"]
    model_inputs = {"Cases only": case_inputs}

    for lag in (3, 6):
        if lag >= horizon:
            field = f"rain_lag_{lag}"
            rows[field] = rain.reindex(target - lag).to_numpy()
            model_inputs[f"Cases + lag {lag} rain"] = [*case_inputs, field]

    # Every model at a given horizon uses the same available rows.
    rows = rows.dropna()
    test_rows = rows.loc[(rows["target"] // 52).between(2024, 2025)]

    for i, row in test_rows.iterrows():
        train = rows.loc[rows["target"] <= row["origin"]]
        test = rows.loc[[i]]
        forecasts = {"Persistence": row["latest_cases"]}

        for name, columns in model_inputs.items():
            model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            model.fit(train[columns], train["actual"])
            forecasts[name] = max(0, model.predict(test[columns])[0])

        for name, forecast in forecasts.items():
            results.append({
                "horizon": horizon,
                "origin_block": int(row["origin"]),
                "target_block": int(row["target"]),
                "model": name,
                "forecast": forecast,
                "actual": row["actual"],
            })

predictions = pd.DataFrame(results)
predictions["absolute_error"] = (
    predictions["forecast"] - predictions["actual"]
).abs()

model_scores = predictions.pivot_table(
    index="horizon", columns="model", values="absolute_error", aggfunc="mean"
)
display(model_scores.round(2))

model,Cases + lag 3 rain,Cases + lag 6 rain,Cases only,Persistence
horizon,,,,
1,32.10,32.53,32.50,33.60
2,43.31,44.20,44.19,46.47
3,54.37,55.43,55.42,59.27
4,NaN,65.94,65.89,69.71


In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

qc, case_column, _ = load_weekly("QUEZON_CITY")
cases = qc[case_column]
rain = qc["RF_NASA"]
origin = cases.index
results = []

for horizon in range(1, 5):
    target = origin + horizon
    season = 2 * np.pi * (target % 52) / 52
    rows = pd.DataFrame({
        "origin": origin,
        "target": target,
        "latest_cases": cases.to_numpy(),
        "last_year_cases": cases.reindex(target - 52).to_numpy(),
        "season_sin": np.sin(season),
        "season_cos": np.cos(season),
        "actual": cases.reindex(target).to_numpy(),
    })

    case_inputs = ["latest_cases", "last_year_cases", "season_sin", "season_cos"]
    model_inputs = {"Cases only": case_inputs}

    for lag in (3, 6):
        if lag >= horizon:
            field = f"rain_lag_{lag}"
            rows[field] = rain.reindex(target - lag).to_numpy()
            model_inputs[f"Cases + lag {lag} rain"] = [*case_inputs, field]

    # Every model at a given horizon uses the same available rows.
    rows = rows.dropna()
    test_rows = rows.loc[(rows["target"] // 52).between(2024, 2025)]

    for i, row in test_rows.iterrows():
        train = rows.loc[rows["target"] <= row["origin"]]
        test = rows.loc[[i]]
        forecasts = {"Persistence": row["latest_cases"]}

        for name, columns in model_inputs.items():
            model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            model.fit(train[columns], train["actual"])
            forecasts[name] = max(0, model.predict(test[columns])[0])

        for name, forecast in forecasts.items():
            results.append({
                "horizon": horizon,
                "origin_block": int(row["origin"]),
                "target_block": int(row["target"]),
                "model": name,
                "forecast": forecast,
                "actual": row["actual"],
            })

predictions = pd.DataFrame(results)
predictions["absolute_error"] = (
    predictions["forecast"] - predictions["actual"]
).abs()

model_scores = predictions.pivot_table(
    index="horizon", columns="model", values="absolute_error", aggfunc="mean"
)
display(model_scores.round(2))

model,Cases + lag 3 rain,Cases + lag 6 rain,Cases only,Persistence
horizon,,,,
1,32.10,32.53,32.50,33.60
2,43.31,44.20,44.19,46.47
3,54.37,55.43,55.42,59.27
4,NaN,65.94,65.89,69.71


## Future-data leakage test

A forecast issued at the end of **2025 source week 30** should stay the same if we change what happens **after** that week. This test runs the saved training cell twice. The first run uses the original observations. The second adds 1,000 to every later dengue count and rainfall value in an in-memory copy. The source workbook is read-only.

The test compares each model and horizon on the same origin and target blocks. All predictions issued through week 30 must match exactly. The future case counts used to score the 15 forecasts issued in week 30 must change by 1,000. That confirms the test changed the scoring data while keeping forecasts independent of future observations. This test covers the notebook's current same-week reporting assumption.

**Run it after saving this notebook.** It reruns the existing training cell identified as `79ab8b3e` and prints a pass message if the checks hold. A mismatch means the training or prediction code used information from after the cutoff. Forecast accuracy and reporting-delay sensitivity are evaluated separately.


In [10]:
import json
import sys
from pathlib import Path
from unittest.mock import patch

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine import upri_data

notebook_path = root / "notebooks" / "04_model_training.ipynb"
saved_notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
training_cell = next(
    (cell for cell in saved_notebook["cells"] if cell.get("id") == "79ab8b3e"),
    None,
)
if training_cell is None or training_cell["cell_type"] != "code":
    raise ValueError("Save notebook 04 with the training cell identified as 79ab8b3e.")

training_code = compile("".join(training_cell["source"]), str(notebook_path), "exec")
qc, case_column, weather = upri_data.load_weekly("QUEZON_CITY")
audit_origin = 2025 * 52 + 30 - 1

# Modify only observations after the audit origin.
changed_qc = qc.copy(deep=True)
changed_qc.loc[changed_qc.index > audit_origin, [case_column, "RF_NASA"]] += 1000


def run_saved_training(frame):
    namespace = {}
    with patch.object(upri_data, "load_weekly", return_value=(frame, case_column, weather)):
        with patch("IPython.display.display"):
            exec(training_code, namespace)
    return namespace["predictions"]


original_predictions = run_saved_training(qc)
changed_predictions = run_saved_training(changed_qc)

keys = ["horizon", "origin_block", "target_block", "model"]
original = original_predictions.loc[
    original_predictions["origin_block"] <= audit_origin
].set_index(keys).sort_index()
changed = changed_predictions.loc[
    changed_predictions["origin_block"] <= audit_origin
].set_index(keys).sort_index()

pd.testing.assert_series_equal(original["forecast"], changed["forecast"], check_exact=True)

at_origin = original.index.get_level_values("origin_block") == audit_origin
assert at_origin.sum() == 15, "Expected all 15 model and horizon combinations at the audit origin."
assert (changed.loc[at_origin, "actual"] - original.loc[at_origin, "actual"]).eq(1000).all()

print(f"PASS: {len(original):,} forecasts through 2025 source week 30 stayed identical.")
print("The scoring case counts changed for all 15 forecasts issued in week 30.")


PASS: 1,266 forecasts through 2025 source week 30 stayed identical.
The scoring case counts changed for all 15 forecasts issued in week 30.


## Rainfall reporting delay sensitivity

Assume a rainfall observation becomes available one source week after it was measured. Keep the case-reporting assumption from the original run. A forecast can use rainfall only if its rainfall block is at least one block before the forecast origin. For a three-week-ahead forecast, lag +3 uses rain from the origin week and is unavailable under this delay. Lag +6 remains available at horizons 1–4.

This cell uses the forecasts from the ridge backtest above. The chosen rainfall lags use fixed historical blocks, and NASA rainfall covers the entire QC source series. The delay removes model and horizon combinations that would have used unavailable rainfall. Predictions for the combinations that remain eligible use the same inputs, fitting rows, and test weeks as before. Run the ridge training cell first, then run this cell.

The table compares each eligible rainfall model with the case-only ridge model on identical forecast origins and target weeks. A negative MAE change favors adding rainfall. Results are separate for 2024 and 2025. The one-week delay is an assumed sensitivity test, and the comparison is retrospective.


In [11]:
import pandas as pd

if "predictions" not in globals() or "qc" not in globals():
    raise RuntimeError("Run the ridge training cell above before running the delay comparison.")
if qc["RF_NASA"].isna().any():
    raise ValueError("Rainfall coverage changed. Recheck matched training rows before reusing forecasts.")

rainfall_lags = {"Cases + lag 3 rain": 3, "Cases + lag 6 rain": 6}
keys = ["horizon", "origin_block", "target_block"]
case_only = predictions.loc[
    predictions["model"].eq("Cases only"), [*keys, "actual", "absolute_error"]
].rename(columns={"absolute_error": "case_only_error"})

rain_candidates = predictions.loc[predictions["model"].isin(rainfall_lags)].copy()
rain_block = rain_candidates["target_block"] - rain_candidates["model"].map(rainfall_lags)
eligible = rain_candidates.loc[rain_block <= rain_candidates["origin_block"] - 1]
paired = eligible.merge(case_only, on=[*keys, "actual"], validate="many_to_one")
if len(paired) != len(eligible):
    raise ValueError("Some rainfall forecasts lack a matching case-only forecast.")

paired["target_year"] = paired["target_block"] // 52
rainfall_delay_scores = paired.groupby(["target_year", "horizon", "model"]).agg(
    forecast_weeks=("target_block", "size"),
    case_only_mae=("case_only_error", "mean"),
    rainfall_mae=("absolute_error", "mean"),
)
rainfall_delay_scores["mae_change"] = (
    rainfall_delay_scores["rainfall_mae"] - rainfall_delay_scores["case_only_mae"]
)
display(rainfall_delay_scores.round(2))


forecast_weeks  case_only_mae  \
target_year horizon model                                               
2024        1       Cases + lag 3 rain              52          24.39   
                    Cases + lag 6 rain              52          24.39   
            2       Cases + lag 3 rain              52          33.53   
                    Cases + lag 6 rain              52          33.53   
            3       Cases + lag 6 rain              52          41.83   
            4       Cases + lag 6 rain              52          50.46   
2025        1       Cases + lag 3 rain              52          40.62   
                    Cases + lag 6 rain              52          40.62   
            2       Cases + lag 3 rain              52          54.85   
                    Cases + lag 6 rain              52          54.85   
            3       Cases + lag 6 rain              52          69.01   
            4       Cases + lag 6 rain              52          81.32   

                                        rainfall_mae  mae_change  
target_year horizon model                                         
2024        1       Cases + lag 3 rain         24.75        0.36  
                    Cases + lag 6 rain         24.39        0.00  
            2       Cases + lag 3 rain         32.93       -0.60  
                    Cases + lag 6 rain         33.56        0.03  
            3       Cases + lag 6 rain         41.87        0.04  
            4       Cases + lag 6 rain         50.33       -0.14  
2025        1       Cases + lag 3 rain         39.45       -1.17  
                    Cases + lag 6 rain         40.66        0.04  
            2       Cases + lag 3 rain         53.69       -1.16  
                    Cases + lag 6 rain         54.83       -0.02  
            3       Cases + lag 6 rain         68.99       -0.02  
            4       Cases + lag 6 rain         81.56        0.24

## Dengue case reporting delay sensitivity

Assume a dengue count becomes available one source week after the week it describes. At a forecast origin of week 30, the newest case input is the week 29 count. The fitted model also stops at week 29 when choosing which historical outcomes it can learn from. Each historical training row uses the same one-week delay for its own latest-case input.

This comparison uses persistence and case-only ridge to isolate the dengue-reporting delay. It keeps the original ridge settings, seasonality inputs, and evaluation weeks. It compares the original same-week reporting assumption with the one-week delay on the same target weeks in each year. A positive MAE change means forecasts were farther off under delayed reporting. This is a retrospective sensitivity test. The actual release dates remain unknown.

Run the ridge training cell above before running this comparison.


In [12]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if "predictions" not in globals() or "qc" not in globals() or "case_column" not in globals():
    raise RuntimeError("Run the ridge training cell above before running this comparison.")


def forecasts_with_case_delay(cases, forecast_origins, *, rain=None):
    """Replay forecasts with both input sources available through origin minus one."""
    case_inputs = ["latest_cases", "last_year_cases", "season_sin", "season_cos"]
    origins = cases.index
    cutoffs = origins - 1
    results = []

    for horizon, evaluation in forecast_origins.groupby("horizon"):
        targets = origins + horizon
        season = 2 * np.pi * (targets % 52) / 52
        rows = pd.DataFrame({
            "target": targets,
            "latest_cases": cases.reindex(cutoffs).to_numpy(),
            "last_year_cases": cases.reindex(targets - 52).to_numpy(),
            "season_sin": np.sin(season),
            "season_cos": np.cos(season),
            "actual": cases.reindex(targets).to_numpy(),
        }, index=origins)
        model_inputs = {"Cases only": case_inputs}
        if rain is not None:
            for lag in (3, 6):
                if lag >= horizon + 1:
                    field = f"rain_lag_{lag}"
                    rows[field] = rain.reindex(targets - lag).to_numpy()
                    model_inputs[f"Cases + lag {lag} rain"] = [*case_inputs, field]
        rows = rows.dropna()

        for origin in evaluation["origin_block"]:
            case_cutoff = origin - 1
            train = rows.loc[rows["target"] <= case_cutoff]
            test = rows.loc[[origin]]
            row = test.iloc[0]

            forecasts = {"Persistence": row["latest_cases"]}
            for name, columns in model_inputs.items():
                model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
                model.fit(train[columns], train["actual"])
                forecasts[name] = max(0, model.predict(test[columns])[0])
            for name, forecast in forecasts.items():
                results.append({
                    "horizon": horizon,
                    "origin_block": origin,
                    "target_block": int(row["target"]),
                    "model": name,
                    "delayed_forecast": forecast,
                })

    return pd.DataFrame(results)


keys = ["horizon", "origin_block", "target_block"]
same_week = predictions.loc[
    predictions["model"].isin(["Persistence", "Cases only"]),
    [*keys, "model", "actual", "absolute_error"],
]
forecast_origins = same_week.loc[same_week["model"].eq("Cases only"), keys]
delayed = forecasts_with_case_delay(qc[case_column], forecast_origins)
matched = same_week.merge(delayed, on=[*keys, "model"], validate="one_to_one")
if len(matched) != len(same_week) or len(matched) != len(delayed):
    raise ValueError("Same-week and delayed forecasts must match on every evaluation row.")

matched["target_year"] = matched["target_block"] // 52
matched["delayed_error"] = (matched["delayed_forecast"] - matched["actual"]).abs()
case_delay_scores = matched.groupby(["target_year", "horizon", "model"]).agg(
    forecast_weeks=("target_block", "size"),
    same_week_mae=("absolute_error", "mean"),
    delayed_mae=("delayed_error", "mean"),
)
case_delay_scores["mae_change"] = (
    case_delay_scores["delayed_mae"] - case_delay_scores["same_week_mae"]
)
display(case_delay_scores.round(2))


forecast_weeks  same_week_mae  delayed_mae  \
target_year horizon model                                                     
2024        1       Cases only               52          24.39        33.53   
                    Persistence              52          23.50        34.00   
            2       Cases only               52          33.53        41.83   
                    Persistence              52          34.00        43.54   
            3       Cases only               52          41.83        50.46   
                    Persistence              52          43.54        49.56   
            4       Cases only               52          50.46        57.81   
                    Persistence              52          49.56        55.85   
2025        1       Cases only               52          40.62        54.85   
                    Persistence              52          43.69        58.94   
            2       Cases only               52          54.85        69.01   
                    Persistence              52          58.94        75.00   
            3       Cases only               52          69.01        81.32   
                    Persistence              52          75.00        89.87   
            4       Cases only               52          81.32        90.24   
                    Persistence              52          89.87        98.88   

                                 mae_change  
target_year horizon model                    
2024        1       Cases only         9.14  
                    Persistence       10.50  
            2       Cases only         8.30  
                    Persistence        9.54  
            3       Cases only         8.63  
                    Persistence        6.02  
            4       Cases only         7.34  
                    Persistence        6.29  
2025        1       Cases only        14.23  
                    Persistence       15.25  
            2       Cases only        14.16  
                    Persistence       16.06  
            3       Cases only        12.30  
                    Persistence       14.87  
            4       Cases only         8.93  
                    Persistence        9.02

## Combined dengue and rainfall reporting delays

Assume both reports become available one source week after the week they describe. An issue in week 30 can use case counts and rainfall through week 29. The ridge fit also uses only training outcomes through week 29. The +3 rainfall lag is eligible at horizons 1–2, while +6 is eligible at horizons 1–4.

Run the dengue case delay cell above first. This comparison adds rainfall to the same case-only ridge model under the combined timing rule. For each year and horizon, every eligible rainfall model is compared with case-only ridge on the same forecast weeks. Negative MAE change favors rainfall. This is a retrospective comparison using an assumed delay.


In [13]:
if "case_delay_scores" not in globals():
    raise RuntimeError("Run the dengue case reporting delay cell above before this comparison.")

joint_delay_forecasts = forecasts_with_case_delay(
    qc[case_column], forecast_origins, rain=qc["RF_NASA"]
)
rain_lags = {"Cases + lag 3 rain": 3, "Cases + lag 6 rain": 6}
forecast_keys = ["horizon", "origin_block", "target_block"]

case_baseline = joint_delay_forecasts.loc[
    joint_delay_forecasts["model"].eq("Cases only"), [*forecast_keys, "delayed_forecast"]
].rename(columns={"delayed_forecast": "case_only_forecast"})
rain_forecasts = joint_delay_forecasts.loc[joint_delay_forecasts["model"].isin(rain_lags)]
actuals = same_week.loc[same_week["model"].eq("Cases only"), [*forecast_keys, "actual"]]

paired_joint = rain_forecasts.merge(case_baseline, on=forecast_keys, validate="many_to_one")
paired_joint = paired_joint.merge(actuals, on=forecast_keys, validate="many_to_one")
if len(paired_joint) != len(rain_forecasts):
    raise ValueError("Some joint-delay forecasts lack matched case-only forecasts or observed counts.")

paired_joint["target_year"] = paired_joint["target_block"] // 52
paired_joint["case_only_error"] = (
    paired_joint["case_only_forecast"] - paired_joint["actual"]
).abs()
paired_joint["rainfall_error"] = (paired_joint["delayed_forecast"] - paired_joint["actual"]).abs()

joint_delay_scores = paired_joint.groupby(["target_year", "horizon", "model"]).agg(
    forecast_weeks=("target_block", "size"),
    case_only_mae=("case_only_error", "mean"),
    rainfall_mae=("rainfall_error", "mean"),
)
joint_delay_scores["mae_change"] = (
    joint_delay_scores["rainfall_mae"] - joint_delay_scores["case_only_mae"]
)
display(joint_delay_scores.round(2))


forecast_weeks  case_only_mae  \
target_year horizon model                                               
2024        1       Cases + lag 3 rain              52          33.53   
                    Cases + lag 6 rain              52          33.53   
            2       Cases + lag 3 rain              52          41.83   
                    Cases + lag 6 rain              52          41.83   
            3       Cases + lag 6 rain              52          50.46   
            4       Cases + lag 6 rain              52          57.81   
2025        1       Cases + lag 3 rain              52          54.85   
                    Cases + lag 6 rain              52          54.85   
            2       Cases + lag 3 rain              52          69.01   
                    Cases + lag 6 rain              52          69.01   
            3       Cases + lag 6 rain              52          81.32   
            4       Cases + lag 6 rain              52          90.24   

                                        rainfall_mae  mae_change  
target_year horizon model                                         
2024        1       Cases + lag 3 rain         32.93       -0.60  
                    Cases + lag 6 rain         33.56        0.03  
            2       Cases + lag 3 rain         40.78       -1.05  
                    Cases + lag 6 rain         41.87        0.04  
            3       Cases + lag 6 rain         50.33       -0.14  
            4       Cases + lag 6 rain         57.41       -0.39  
2025        1       Cases + lag 3 rain         53.69       -1.16  
                    Cases + lag 6 rain         54.83       -0.02  
            2       Cases + lag 3 rain         67.96       -1.05  
                    Cases + lag 6 rain         68.99       -0.02  
            3       Cases + lag 6 rain         81.56        0.24  
            4       Cases + lag 6 rain         90.14       -0.10

## High-case weeks under both reporting delays

For each forecast, define a high-case target as a week whose actual count reaches the 90th percentile of the QC counts available at that origin. With the one-week case reporting delay, the threshold uses counts only through the week before the origin. The observed target count identifies the evaluation group after the forecast has been made.

Compare case-only ridge with the same model plus +3 rainfall at the one- and two-week horizons. Both models use the same forecast rows. A negative MAE change favors rainfall. Negative bias indicates underprediction. This threshold is a research definition of high case counts. Results are exploratory, particularly when only a few weeks qualify.


In [ ]:
from IPython.display import display

if "paired_joint" not in globals():
    raise RuntimeError("Run the combined reporting-delay cell above before this evaluation.")

rain3 = paired_joint.loc[paired_joint["model"].eq("Cases + lag 3 rain")].copy()
known_case_thresholds = qc[case_column].expanding().quantile(0.90)
rain3["high_case_threshold"] = (rain3["origin_block"] - 1).map(known_case_thresholds)
if rain3["high_case_threshold"].isna().any():
    raise ValueError("A forecast origin is missing its historical high-case threshold.")

high_case = rain3.loc[rain3["actual"] >= rain3["high_case_threshold"]].copy()
high_case["case_only_signed_error"] = high_case["case_only_forecast"] - high_case["actual"]
high_case["rainfall_signed_error"] = high_case["delayed_forecast"] - high_case["actual"]

high_case_scores = high_case.groupby(["target_year", "horizon"]).agg(
    high_case_weeks=("target_block", "size"),
    case_only_mae=("case_only_error", "mean"),
    rainfall_mae=("rainfall_error", "mean"),
    case_only_bias=("case_only_signed_error", "mean"),
    rainfall_bias=("rainfall_signed_error", "mean"),
)
high_case_scores["mae_change"] = high_case_scores["rainfall_mae"] - high_case_scores["case_only_mae"]
display(high_case_scores.round(2))


In [14]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

qc, case_column, _ = load_weekly("QUEZON_CITY")
cases = qc[case_column]
origin = cases.index

case_inputs = ["latest_cases", "last_year_cases", "season_sin", "season_cos"]
model_inputs = {
    "Cases + seasonality": case_inputs,
    "Cases + seasonality + 6-week rain": [*case_inputs, "rain_lag_6"],
}

results = []

for horizon in range(1, 5):
    target = origin + horizon
    season = 2 * np.pi * (target % 52) / 52

    rows = pd.DataFrame({
        "origin": origin,
        "target": target,
        "latest_cases": cases.to_numpy(),
        "last_year_cases": cases.reindex(target - 52).to_numpy(),
        "season_sin": np.sin(season),
        "season_cos": np.cos(season),
        "rain_lag_6": qc["RF_NASA"].reindex(target - 6).to_numpy(),
        "actual": cases.reindex(target).to_numpy(),
    }).dropna()

    test_rows = rows.loc[(rows["target"] // 52).between(2024, 2025)]

    for i, row in test_rows.iterrows():
        # Fit using only case outcomes through the forecast origin.
        train = rows.loc[rows["target"] <= row["origin"]]
        test = rows.loc[[i]]

        for name, inputs in model_inputs.items():
            model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            model.fit(train[inputs], train["actual"])
            forecast = max(0, model.predict(test[inputs])[0])

            results.append({
                "Weeks ahead": horizon,
                "Model": name,
                "Absolute error": abs(forecast - row["actual"]),
            })

lag6_predictions = pd.DataFrame(results)
lag6_scores = lag6_predictions.groupby(["Weeks ahead", "Model"]).agg(
    Forecasts=("Absolute error", "size"),
    MAE=("Absolute error", "mean"),
)

display(lag6_scores.round(2))

Forecasts    MAE
Weeks ahead Model                                              
1           Cases + seasonality                      104  32.50
            Cases + seasonality + 6-week rain        104  32.53
2           Cases + seasonality                      104  44.19
            Cases + seasonality + 6-week rain        104  44.20
3           Cases + seasonality                      104  55.42
            Cases + seasonality + 6-week rain        104  55.43
4           Cases + seasonality                      104  65.89
            Cases + seasonality + 6-week rain        104  65.94

## Carvajal-style Random Forest comparison for Quezon City

Fit two weather-only Random Forests to weekly log cases per 1,000 QC residents. One uses weather from the dengue week and the other uses lags selected on 2021–2024 outcomes. Both are evaluated on the same 52 source weeks of 2025. The 2021 COVID-flagged records remain in the training period. Calendar alignment follows the workbook's source-block order.

Scan 0–25 source weeks for each weather field with Pearson correlation. Select the largest absolute correlation and use the shorter lag if tied. This is the rule for the QC experiment. Only outcomes in 2021–2024 inform the lag search. Weather observations from before 2021 can supply the earlier weather for those outcomes. Both models use all complete matched rows.

The paper used 1,000 trees in R. This QC adaptation uses scikit-learn, a fixed seed, bootstrap sampling, one candidate feature per split and a minimum leaf size of five, inspired by R regression defaults. R and scikit-learn can produce different trees. The QC weather fields are NASA rainfall, ERA5-Land humidity, and ERA5-Land mean, minimum and maximum temperature. The flood pilot remains a source audit with uncertain historical coverage.

Run the cell below to rebuild the target from the saved population calculation in notebook 01. The results report the paper's log-incidence error scale and the corresponding back-transformed errors in QC cases. These are retrospective target-week predictions. The same-week model uses measured target-week weather, and NASA IMERG Final Run typically arrives later than the historical observation week. Compare ridge and Random Forest at 1–4 weeks ahead in a separate forecast experiment with matched origins and input availability.

[Carvajal et al., methods and results](https://link.springer.com/article/10.1186/s12879-018-3066-0) · [R randomForest regression defaults](https://search.r-project.org/CRAN/refmans/randomForest/html/randomForest.html) · [Population and source audit](01_standardization.ipynb)

### Verified reference result

Using 208 training weeks and 52 test weeks, same-week weather MAE was 1.187 on the log-incidence scale and 150.53 cases after back-transformation. Lagged weather MAE was 1.040 on the log-incidence scale and 138.63 cases. These figures summarize all five weather fields together. Changing only the 2025 observed outcomes left the fitted predictions and selected lags identical.


In [15]:
import json
import sys
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

rf_qc, rf_case_column, _ = load_weekly("QUEZON_CITY")
standardization_path = root / "notebooks" / "01_standardization.ipynb"
standardization = json.loads(standardization_path.read_text(encoding="utf-8"))
population_source = next(
    "".join(cell["source"]) for cell in standardization["cells"]
    if cell.get("id") == "01-qc-population"
)
population_context = {"standardized_qc": rf_qc, "case_column": rf_case_column}
with patch("IPython.display.display"):
    exec(compile(population_source, str(standardization_path), "exec"), population_context)
rf_incidence = population_context["qc_incidence"]
rf_weather_fields = [
    "RF_NASA", "RH_ERA5-Land", "TempMin_ERA5-Land",
    "Temp_ERA5-Land", "TempMax_ERA5-Land",
]


def compare_weather_timing(
    frame, weather_fields, case_column, *, train_start=2021, evaluation_year=2025,
    lagged_flood=None,
):
    target = frame["log_cases_per_1000"]
    train_years = frame["YR"].between(train_start, evaluation_year - 1)
    test_year = frame["YR"].eq(evaluation_year)
    same_week = frame[weather_fields]
    lagged = pd.DataFrame(index=frame.index)
    selected = []

    for field in weather_fields:
        correlations = pd.Series({
            lag: same_week[field].shift(lag).loc[train_years].corr(target.loc[train_years])
            for lag in range(26)
        }).dropna()
        if correlations.empty:
            raise ValueError(f"No usable training correlations for {field}.")
        lag = int(correlations.abs().idxmax())
        lagged[field] = same_week[field].shift(lag)
        selected.append({
            "Weather field": field, "Lag (source weeks)": lag,
            "Training Pearson r": correlations.loc[lag],
        })

    complete = (
        same_week.notna().all(axis=1) & lagged.notna().all(axis=1) & target.notna()
    )
    training = train_years & complete
    evaluation = test_year & complete
    predictions = frame.loc[evaluation, ["YR", "WN", case_column]].copy()
    predictions["actual_log_incidence"] = target.loc[evaluation]
    scores = []

    feature_sets = {"Same-week weather": same_week, "Lagged weather": lagged}
    if lagged_flood is not None:
        if not lagged_flood.index.equals(frame.index):
            raise ValueError("Lagged flood evidence must align with the target source weeks.")
        feature_sets["Lagged weather + flood evidence (lag 6)"] = lagged.assign(
            flood_report_lag_6=lagged_flood,
        )

    for name, features in feature_sets.items():
        model = RandomForestRegressor(
            n_estimators=1_000, random_state=42, bootstrap=True,
            max_features=1, min_samples_leaf=5,
        )
        model.fit(features.loc[training], target.loc[training])
        predicted_log = model.predict(features.loc[evaluation])
        predicted_cases = np.exp(predicted_log) * frame.loc[evaluation, "qc_population"] / 1_000
        log_error = predicted_log - target.loc[evaluation]
        case_error = predicted_cases - frame.loc[evaluation, case_column]
        predictions[f"{name} predicted log incidence"] = predicted_log
        predictions[f"{name} predicted cases"] = predicted_cases
        scores.append({
            "Random Forest inputs": name,
            "Training weeks": int(training.sum()),
            "Test weeks": int(evaluation.sum()),
            "MAE (log incidence)": log_error.abs().mean(),
            "RMSE (log incidence)": np.sqrt(log_error.pow(2).mean()),
            "MAE (cases)": case_error.abs().mean(),
            "RMSE (cases)": np.sqrt(case_error.pow(2).mean()),
        })

    return pd.DataFrame(selected), pd.DataFrame(scores), predictions


rf_lags, rf_timing_scores, rf_timing_predictions = compare_weather_timing(
    rf_incidence, rf_weather_fields, rf_case_column
)
print("Training 2021–2024, testing 2025")
display(rf_lags.round(3))
display(rf_timing_scores.round(3))

print("Training 2010–2013, testing 2014")
rf_2014_lags, rf_2014_scores, rf_2014_predictions = compare_weather_timing(
    rf_incidence, rf_weather_fields, rf_case_column,
    train_start=2010, evaluation_year=2014,
)
display(rf_2014_lags.round(3))
display(rf_2014_scores.round(3))


,Weather field,Lag (source weeks),Training Pearson r
0,RF_NASA,6,0.370
1,RH_ERA5-Land,25,-0.535
2,TempMin_ERA5-Land,13,0.539
3,Temp_ERA5-Land,19,0.532
4,TempMax_ERA5-Land,22,0.564


,Random Forest inputs,Training weeks,Test weeks,MAE (log incidence),RMSE (log incidence),MAE (cases),RMSE (cases)
0,Same-week weather,208,52,1.187,1.334,150.532,186.666
1,Lagged weather,208,52,1.040,1.212,138.633,177.125


### Matching the study's five-year design from 2010

Training uses 2010–2013, so 2014 is the fifth year and the test year. This follows Carvajal et al.'s four-year training and one-year test design, shifted forward one year. The test year comes from the design before looking at its errors.

The training period has 208 candidate outcomes. The QC workbook begins in 2010, so early outcomes can lack earlier weather. Both forests train on the same rows with complete features at the selected lags. The displayed training count reflects that shared coverage. Lag screening uses only 2010–2013 outcomes and the available weather pairs for each lag. The 2025 comparison above remains available for period sensitivity.

[Carvajal et al., methods](https://link.springer.com/article/10.1186/s12879-018-3066-0)

#### Verified 2014 results

Both models used 191 matched training weeks and all 52 test weeks. The longest selected lag was 17 source weeks, so the first 17 training targets lacked the required earlier weather.

| Random Forest inputs | MAE (log incidence) | RMSE (log incidence) | MAE (cases) | RMSE (cases) |
|---|---:|---:|---:|---:|
| Same-week weather | 0.710 | 0.856 | 41.98 | 53.43 |
| Lag selected per field | 0.747 | 0.876 | 47.90 | 64.37 |

The training-period screen selected lag 0 for rainfall and relative humidity, and lags 12, 14 and 17 for minimum, mean and maximum temperature. Same-week weather scored better than the selected-lag setup in 2014. The two test years use different training periods and coverage, so this is evidence of sensitivity to the period. It is a retrospective weather-timing comparison.


## Carvajal-style Random Forest with six-week flood evidence

Run the Carvajal-style Random Forest cell above first. This comparison keeps the original 2010–2013 → 2014 and 2021–2024 → 2025 splits, log-incidence target, training-only weather lag selection, and 1,000-tree settings. Compare **Lagged weather** with **Lagged weather + flood evidence (lag 6)** to measure the effect of adding the flood input on identical weeks. The flood lag is fixed at six source weeks, following the paper's selected flood lag.

A value of 1 means the QC archive contains a dated official or news report six weeks before the target. Missing stays missing, and the Random Forest uses its built-in handling of missing values. The feature distinguishes documented flooding from unknown status. The coverage table shows how often the model receives a documented report in each split.

This is a retrospective experiment using the provisional workbook calendar and the collected report inventory. Its errors measure the value of the available report signal under this setup. Archive coverage varies and can affect that signal. A prospective test would also need verified release dates for both flood reports and weather.

[Flood inventory and week alignment](../data/reference/upri_noah/qc-flood-reports.md) · [Carvajal et al., results and flood lag](https://link.springer.com/article/10.1186/s12879-018-3066-0)


In [18]:
if "compare_weather_timing" not in globals():
    raise RuntimeError("Run the Carvajal-style Random Forest comparison above first.")

rf_flood_path = root / "data/processed/upri_noah/qc-flood-weekly-evidence.csv"
rf_flood_columns = ["YR", "WN", "reported_qc_flood_provisional"]
rf_flood_data = pd.read_csv(rf_flood_path, usecols=rf_flood_columns)
rf_flood_keys = pd.MultiIndex.from_frame(rf_flood_data[["YR", "WN"]])
rf_incidence_keys = pd.MultiIndex.from_frame(rf_incidence[["YR", "WN"]])
if not rf_flood_keys.is_unique or set(rf_flood_keys) != set(rf_incidence_keys):
    raise ValueError("Flood evidence and dengue data must have the same unique source weeks.")

rf_flood_reported = (
    rf_flood_data.set_index(["YR", "WN"])["reported_qc_flood_provisional"]
    .reindex(rf_incidence_keys)
    .set_axis(rf_incidence.index)
)
if not rf_flood_reported.dropna().eq(1).all():
    raise ValueError("The flood input must contain only documented reports (1) and unknowns.")
rf_flood_lag_6 = rf_flood_reported.shift(6)
flood_rf_results = {}
rf_flood_weather_lags = {}
coverage = []
for evaluation_year in (2014, 2025):
    train_start = evaluation_year - 4
    lags, scores, predictions = compare_weather_timing(
        rf_incidence, rf_weather_fields, rf_case_column,
        train_start=train_start, evaluation_year=evaluation_year,
        lagged_flood=rf_flood_lag_6,
    )
    flood_rf_results[evaluation_year] = scores, predictions
    rf_flood_weather_lags[evaluation_year] = lags
    training = rf_incidence["YR"].between(train_start, evaluation_year - 1)
    training &= rf_incidence["log_cases_per_1000"].notna()
    training &= rf_incidence[rf_weather_fields].notna().all(axis=1)
    for field, lag in zip(lags["Weather field"], lags["Lag (source weeks)"]):
        training &= rf_incidence[field].shift(lag).notna()
    coverage.append({
        "Test year": evaluation_year,
        "Training rows with report at lag 6": rf_flood_lag_6.loc[training].count(),
        "Test rows with report at lag 6": rf_flood_lag_6.loc[predictions.index].count(),
    })

    print(f"Training {train_start}–{evaluation_year - 1}, testing {evaluation_year}")
    display(scores.round(3))

rf_flood_coverage = pd.DataFrame(coverage)
display(rf_flood_coverage)


Training 2010–2013, testing 2014


,Random Forest inputs,Training weeks,Test weeks,MAE (log incidence),RMSE (log incidence),MAE (cases),RMSE (cases)
0,Same-week weather,191,52,0.710,0.856,41.977,53.428
1,Lagged weather,191,52,0.747,0.876,47.904,64.366
2,Lagged weather + flood evidence (lag 6),191,52,0.749,0.883,46.155,60.427


Training 2021–2024, testing 2025


,Random Forest inputs,Training weeks,Test weeks,MAE (log incidence),RMSE (log incidence),MAE (cases),RMSE (cases)
0,Same-week weather,208,52,1.187,1.334,150.532,186.666
1,Lagged weather,208,52,1.040,1.212,138.633,177.125
2,Lagged weather + flood evidence (lag 6),208,52,1.047,1.213,139.724,177.682


,Test year,Training rows with report at lag 6,Test rows with report at lag 6
0,2014,15,4
1,2025,14,6


## Matched weekly forecast comparison — Random Forest and ridge

Compare case-only ridge and Random Forest on the same 2024–2025 QC target weeks and forecast origins at horizons 1–4. Both receive the latest reported count, the count for the corresponding source week a year earlier, and seasonal terms. Both fit directly to QC case counts and refit at each origin. Training outcomes stop at the origin, following the same retrospective case-reporting assumption as the earlier ridge comparison. The Random Forest uses the same 1,000-tree settings as the Carvajal-style QC experiment. These are retrospective results.

### Six-week rainfall availability

At horizons 1–4, rainfall measured six source weeks before the target was measured only 5, 4, 3, or 2 source weeks before the issue. NASA IMERG Final Run usually arrives around 3.5 months after observation. The workbook provides complete Final Run observations and omits their original release timestamps. The matched comparison therefore includes only case-based models. A rainfall comparison at issue time needs a rainfall series whose release can be checked and whose weekly alignment is verified. The workbook's PAGASA reference series is sparser and its provenance does not establish release times.

[NASA IMERG latency](https://gpm.nasa.gov/resources/faq/what-determines-latency-imerg) · [Workbook source notes](../data/raw/upri_noah/README.md)


In [16]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.upri_data import load_weekly

matched_qc, matched_case_column, _ = load_weekly("QUEZON_CITY")
matched_cases = matched_qc[matched_case_column]
matched_inputs = ["latest_cases", "last_year_cases", "season_sin", "season_cos"]


def matched_forecast_rows(cases, horizon):
    origin = cases.index
    target = origin + horizon
    season = 2 * np.pi * (target % 52) / 52
    return pd.DataFrame({
        "origin": origin,
        "target": target,
        "latest_cases": cases.to_numpy(),
        "last_year_cases": cases.reindex(target - 52).to_numpy(),
        "season_sin": np.sin(season),
        "season_cos": np.cos(season),
        "actual": cases.reindex(target).to_numpy(),
    }).dropna()


def fit_matched_forecast(rows, test, horizon):
    origin = int(test["origin"].iloc[0])
    training = rows.loc[rows["target"] <= origin]
    models = {
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "Random Forest": RandomForestRegressor(
            n_estimators=1_000, random_state=42, bootstrap=True,
            max_features=1, min_samples_leaf=5, n_jobs=1,
        ),
    }
    forecasts = []
    for name, model in models.items():
        model.fit(training[matched_inputs], training["actual"])
        forecast = max(0, model.predict(test[matched_inputs])[0])
        forecasts.append({
            "horizon": horizon,
            "origin_block": origin,
            "target_block": int(test["target"].iloc[0]),
            "model": name,
            "forecast": forecast,
            "actual": test["actual"].iloc[0],
        })
    return forecasts


def replay_matched_horizon(horizon):
    rows = matched_forecast_rows(matched_cases, horizon)
    test_rows = rows.loc[(rows["target"] // 52).between(2024, 2025)]
    forecasts = []
    for i in test_rows.index:
        forecasts.extend(fit_matched_forecast(rows, rows.loc[[i]], horizon))
    print(f"Horizon {horizon} complete: {len(test_rows)} forecasts per model.", flush=True)
    return pd.DataFrame(forecasts)


print("Training matched case-only models for horizons 1–4.", flush=True)
with ThreadPoolExecutor(max_workers=4) as pool:
    rf_matched_predictions = pd.concat(
        list(pool.map(replay_matched_horizon, range(1, 5))), ignore_index=True
    )

rf_matched_predictions["target_year"] = rf_matched_predictions["target_block"] // 52
rf_matched_predictions["error"] = (
    rf_matched_predictions["forecast"] - rf_matched_predictions["actual"]
)
rf_matched_predictions["absolute_error"] = rf_matched_predictions["error"].abs()
rf_matched_predictions["squared_error"] = rf_matched_predictions["error"].pow(2)

rf_matched_scores = rf_matched_predictions.groupby(["horizon", "model"]).agg(
    forecast_weeks=("target_block", "size"),
    mae=("absolute_error", "mean"),
    mse=("squared_error", "mean"),
)
rf_matched_scores["rmse"] = np.sqrt(rf_matched_scores.pop("mse"))
display(rf_matched_scores.round(2))


Training matched case-only models for horizons 1–4.
Horizon 4 complete: 104 forecasts per model.
Horizon 1 complete: 104 forecasts per model.
Horizon 3 complete: 104 forecasts per model.
Horizon 2 complete: 104 forecasts per model.


forecast_weeks    mae    rmse
horizon model                                       
1       Random Forest             104  47.44   81.63
        Ridge                     104  32.50   50.43
2       Random Forest             104  53.82   90.83
        Ridge                     104  44.19   69.12
3       Random Forest             104  59.78   99.22
        Ridge                     104  55.42   81.92
4       Random Forest             104  64.16  104.76
        Ridge                     104  65.89   92.01

## Matched six-week NASA rainfall comparison — retrospective

Run the matched case-only comparison above first. The third model uses the same Random Forest settings, case inputs, training rows, and forecast rows as case-only Random Forest, plus NASA IMERG Final Run rainfall from six source weeks before the target week.

This experiment uses the historical Final Run rainfall values as a retrospective input. The values were generally released after the forecast origins. The comparison measures whether the finalized rainfall series improves retrospective accuracy. A comparison that requires rainfall as available on the issue date remains a separate question.


In [17]:
if "matched_forecast_rows" not in globals():
    raise RuntimeError("Run the matched case-only Random Forest comparison above first.")


def add_final_run_rainfall(rows, rainfall):
    rain_block = rows["target"] - 6
    result = rows.assign(
        rain_lag_6=rain_block.map(rainfall),
        rain_block=rain_block,
    )
    if result["rain_lag_6"].isna().any():
        raise ValueError("Rainfall coverage changed. Recheck matched rows before comparing models.")
    if result["rain_block"].gt(result["origin"]).any():
        raise ValueError("A rainfall observation occurs after its forecast origin.")
    return result


def fit_final_run_rainfall(rows, test, horizon):
    origin = int(test["origin"].iloc[0])
    training = rows.loc[rows["target"] <= origin]
    inputs = [*matched_inputs, "rain_lag_6"]
    model = RandomForestRegressor(
        n_estimators=1_000, random_state=42, bootstrap=True,
        max_features=1, min_samples_leaf=5, n_jobs=1,
    )
    model.fit(training[inputs], training["actual"])
    forecast = max(0, model.predict(test[inputs])[0])
    return {
        "horizon": horizon,
        "origin_block": origin,
        "target_block": int(test["target"].iloc[0]),
        "model": "Random Forest + NASA rain (lag 6)",
        "forecast": forecast,
        "actual": test["actual"].iloc[0],
    }


def replay_final_run_horizon(horizon):
    rows = add_final_run_rainfall(
        matched_forecast_rows(matched_cases, horizon), matched_qc["RF_NASA"]
    )
    test_rows = rows.loc[(rows["target"] // 52).between(2024, 2025)]
    forecasts = []
    for position, i in enumerate(test_rows.index, start=1):
        forecasts.append(fit_final_run_rainfall(rows, rows.loc[[i]], horizon))
        if position % 26 == 0:
            print(f"NASA rainfall horizon {horizon}: {position}/{len(test_rows)} forecasts.", flush=True)
    return pd.DataFrame(forecasts)


print("Training matched Random Forest with six-week NASA rainfall.", flush=True)
with ThreadPoolExecutor(max_workers=4) as pool:
    rf_final_run_predictions = pd.concat(
        list(pool.map(replay_final_run_horizon, range(1, 5))), ignore_index=True
    )

rf_final_run_predictions["target_year"] = rf_final_run_predictions["target_block"] // 52
rf_final_run_predictions["error"] = (
    rf_final_run_predictions["forecast"] - rf_final_run_predictions["actual"]
)
rf_final_run_predictions["absolute_error"] = rf_final_run_predictions["error"].abs()
rf_final_run_predictions["squared_error"] = rf_final_run_predictions["error"].pow(2)

rf_final_run_scores = rf_final_run_predictions.groupby(["horizon", "model"]).agg(
    forecast_weeks=("target_block", "size"),
    mae=("absolute_error", "mean"),
    mse=("squared_error", "mean"),
)
rf_final_run_scores["rmse"] = np.sqrt(rf_final_run_scores.pop("mse"))
display(rf_final_run_scores.round(2))


Training matched Random Forest with six-week NASA rainfall.
NASA rainfall horizon 2: 26/104 forecasts.
NASA rainfall horizon 3: 26/104 forecasts.
NASA rainfall horizon 4: 26/104 forecasts.
NASA rainfall horizon 1: 26/104 forecasts.
NASA rainfall horizon 4: 52/104 forecasts.
NASA rainfall horizon 2: 52/104 forecasts.
NASA rainfall horizon 3: 52/104 forecasts.
NASA rainfall horizon 1: 52/104 forecasts.
NASA rainfall horizon 4: 78/104 forecasts.
NASA rainfall horizon 3: 78/104 forecasts.
NASA rainfall horizon 2: 78/104 forecasts.
NASA rainfall horizon 1: 78/104 forecasts.
NASA rainfall horizon 4: 104/104 forecasts.
NASA rainfall horizon 2: 104/104 forecasts.
NASA rainfall horizon 3: 104/104 forecasts.
NASA rainfall horizon 1: 104/104 forecasts.


,,forecast_weeks,mae,rmse
horizon,model,,,
1,Random Forest + NASA rain (lag 6),104,50.67,86.47
2,Random Forest + NASA rain (lag 6),104,55.97,94.45
3,Random Forest + NASA rain (lag 6),104,60.99,101.60
4,Random Forest + NASA rain (lag 6),104,65.36,106.31


## Verified matched forecast results

Each model was scored on 104 identical target weeks per horizon. Errors are in reported cases.

| Weeks ahead | Ridge MAE | Case-only RF MAE | RF with NASA rainfall MAE | Ridge RMSE | Case-only RF RMSE | RF with NASA rainfall RMSE |
|---:|---:|---:|---:|---:|---:|---:|
| 1 | 32.50 | 47.44 | 50.67 | 50.43 | 81.63 | 86.47 |
| 2 | 44.19 | 53.82 | 55.97 | 69.12 | 90.83 | 94.45 |
| 3 | 55.42 | 59.78 | 60.99 | 81.92 | 99.22 | 101.60 |
| 4 | 65.89 | 64.16 | 65.36 | 92.01 | 104.76 | 106.31 |

The rainfall column is retrospective and conditional on the workbook's Final Run values. The verification reproduced the prior ridge predictions exactly, checked matched evaluation rows, and confirmed that perturbing observations after the audit origin leaves its predictions unchanged.


## Seasonal Random Forest versus the existing XGBoost projection

This comparison issues all 52 weekly predictions before each test year. For 2024, train on 2010–2023. For 2025, train on 2010–2024. Earlier 2018–2023 folds are diagnostics because the XGBoost variant was selected using those years. The 2024–2025 results are retrospective.

Match the existing XGBoost weather/all-years variant. Both use the same four seasonal terms and the same rainfall, temperature and humidity lags of 1, 2 and 4 source weeks. Training uses earlier observed weather. Prediction uses the same typical-weather analog drawn from past years, with all inputs frozen before the target year. The Random Forest settings are fixed at 1,000 trees, one candidate feature per split, a minimum leaf size of five, and seed 42. XGBoost is refitted in the current environment using the variant selected in the saved validation artifact. The saved artifact was produced with a different software environment, and its scores differ slightly from this rerun.

MAE measures the average size of a miss. RMSE gives large misses more weight. Bias is forecast minus actual, so negative bias means underprediction. This is the year-ahead seasonal projection experiment. The 1–4-week comparison earlier in this notebook uses recent case counts available at each weekly origin.

[Seasonal training script](../scripts/train_qc_seasonal.py) · [Seasonal feature and weather analog code](../services/engine/domain/seasonal_forecast.py)

### Verified 2024–2025 results

| Evaluation scope | Weeks per model | Random Forest MAE | XGBoost MAE |
|---|---:|---:|---:|
| 2024 | 52 | 78.41 | 79.53 |
| 2025 | 52 | 111.96 | 112.85 |
| Combined | 104 | 95.18 | 96.19 |

The combined RMSE was 144.23 for Random Forest and 145.53 for XGBoost. Combined bias was −91.25 and −92.33 cases per week, respectively. The difference between the two models was small relative to their underprediction. Verification matched all 416 target weeks to the archived fold records and reproduced the current seasonal XGBoost predictions with the same feature matrices used by Random Forest in the two main years.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from threadpoolctl import threadpool_limits

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))

from services.engine.domain import seasonal_forecast
from services.engine.upri_data import load_weekly

seasonal_qc, seasonal_case, _ = load_weekly("QUEZON_CITY")
reference_path = root / "data" / "processed" / "upri_noah" / "qc-seasonal-validation.json"
seasonal_reference = json.loads(reference_path.read_text(encoding="utf-8"))
if seasonal_reference["selection"]["xgboostVariant"] != "weather_all_years":
    raise ValueError("The saved XGBoost variant changed. Review the matched feature policy.")
seasonal_xgb_records = pd.DataFrame(seasonal_reference["records"]).set_index("block")


def seasonal_comparison_features(history, target_weeks):
    train_features = [*seasonal_forecast.seasonal_features(history["WN"]).T]
    forecast_features = [*seasonal_forecast.seasonal_features(target_weeks).T]
    analogs = {
        (week, lag): seasonal_forecast.weather_analog(
            history, (int(week) - lag - 1) % 52 + 1
        )
        for week in set(target_weeks)
        for lag in seasonal_forecast.LAGS
    }
    for field in seasonal_forecast.WEATHER:
        for lag in seasonal_forecast.LAGS:
            train_features.append(history[field].shift(lag).to_numpy())
            forecast_features.append([analogs[week, lag][field] for week in target_weeks])
    return np.column_stack(train_features), np.column_stack(forecast_features)


seasonal_predictions = []
for test_year in range(2018, 2026):
    history = seasonal_qc.loc[seasonal_qc["YR"] < test_year]
    target = seasonal_qc.loc[seasonal_qc["YR"].eq(test_year)]
    reference = seasonal_xgb_records.loc[target.index]

    if not (
        np.array_equal(reference["observed"], target[seasonal_case])
        and np.array_equal(reference["week"], target["WN"])
        and reference["year"].eq(test_year).all()
        and reference["lastTrainingBlock"].eq(history.index.max()).all()
    ):
        raise ValueError(f"Saved XGBoost records and QC inputs do not match for {test_year}.")

    train_features, forecast_features = seasonal_comparison_features(
        history, target["WN"].astype(int).tolist()
    )
    eligible = np.isfinite(train_features).all(axis=1) & history[seasonal_case].notna()
    forest = RandomForestRegressor(
        n_estimators=1_000, random_state=42, bootstrap=True,
        max_features=1, min_samples_leaf=5, n_jobs=1,
    )
    forest.fit(train_features[eligible], history.loc[eligible, seasonal_case])
    with threadpool_limits(limits=2):
        xgb_engine = seasonal_forecast.train_engine(history)
        xgb_engine["xgboostVariant"] = seasonal_reference["selection"]["xgboostVariant"]
        xgb_forecast = seasonal_forecast.predict_weeks(
            xgb_engine, target["WN"].astype(int).tolist()
        )["seasonal_xgboost"]
    seasonal_predictions.append(pd.DataFrame({
        "year": test_year,
        "source_week": target["WN"],
        "actual_cases": target[seasonal_case],
        "Random Forest": forest.predict(forecast_features),
        "XGBoost": xgb_forecast,
    }, index=target.index.rename("source_block")))

seasonal_rf_xgb_predictions = pd.concat(seasonal_predictions)
seasonal_errors = seasonal_rf_xgb_predictions.reset_index().melt(
    id_vars=["source_block", "year", "source_week", "actual_cases"],
    value_vars=["Random Forest", "XGBoost"], var_name="model", value_name="forecast",
)
seasonal_errors["error"] = seasonal_errors["forecast"] - seasonal_errors["actual_cases"]
seasonal_errors["absolute_error"] = seasonal_errors["error"].abs()
seasonal_errors["squared_error"] = seasonal_errors["error"].pow(2)


def summarize_seasonal_errors(errors, by):
    scores = errors.groupby(by).agg(
        weeks=("error", "size"), mae=("absolute_error", "mean"),
        mse=("squared_error", "mean"), bias=("error", "mean"),
    )
    scores["rmse"] = np.sqrt(scores.pop("mse"))
    return scores[["weeks", "mae", "rmse", "bias"]]


seasonal_rf_xgb_yearly = summarize_seasonal_errors(seasonal_errors, ["year", "model"])
seasonal_rf_xgb_main = summarize_seasonal_errors(
    seasonal_errors.loc[seasonal_errors["year"].between(2024, 2025)], "model"
)
display(seasonal_rf_xgb_yearly.round(2))
display(seasonal_rf_xgb_main.round(2))


## Previous-year total distributed by historical weekly shares

This year-ahead benchmark carries forward the previous source year's total reported QC cases. It distributes that total across 52 source weeks using the average weekly shares from complete years available before the test year. Each historical year contributes equally to the seasonal shape. The prior-year total and weekly shares are frozen before the forecast year. This retrospective calculation assumes a completed year of reports is available at the cutoff. Actual reporting vintages remain unknown.

Compare it with the matched Random Forest and XGBoost yearly projections on all 52 weeks in 2024 and 2025. A target week counts as high-case when its observed count reaches the 90th percentile of the weekly counts from before that test year. This threshold is a research definition. It uses only information from the training period, and scoring uses the same high-case weeks for all three models.

Run the **Seasonal Random Forest versus the existing XGBoost projection** cell above first. The tables report MAE and bias in cases. Negative bias means underprediction.

### Verified 2024–2025 results

| Evaluation | Weeks per model | Previous-year total MAE | Random Forest MAE | XGBoost MAE |
|---|---:|---:|---:|---:|
| 2024 | 52 | 88.05 | 78.41 | 79.53 |
| 2025 | 52 | 84.88 | 111.96 | 112.85 |
| Combined | 104 | 86.47 | 95.18 | 96.19 |
| High-case weeks in 2024 | 16 | 233.69 | 206.30 | 205.84 |
| High-case weeks in 2025 | 20 | 153.05 | 217.59 | 217.30 |

The benchmark's combined bias was −68.25 cases per week, compared with −91.25 for Random Forest and −92.33 for XGBoost. It improved the combined score because it did substantially better in 2025. Both tree models performed better in 2024, when the previous-year total supplied too low a starting point. During high-case weeks all three models underpredicted on average in both years. The matching checks confirmed all methods used the same target weeks and high-case thresholds and that the benchmark conserved its prior-year totals.


In [ ]:
if "seasonal_rf_xgb_predictions" not in globals():
    raise RuntimeError("Run the seasonal Random Forest versus XGBoost cell above first.")


def previous_total_by_week(history, test_year):
    """Anchor the weekly projection to the last completed year's case total."""
    complete_years = (
        history.pivot(index="YR", columns="WN", values=seasonal_case)
        .reindex(columns=range(1, 53))
        .dropna()
    )
    if test_year - 1 not in complete_years.index:
        raise ValueError(f"Previous year {test_year - 1} lacks complete weekly counts.")
    annual_totals = complete_years.sum(axis=1)
    if annual_totals.le(0).any():
        raise ValueError("Complete source years must have positive annual case totals.")
    average_share = complete_years.div(annual_totals, axis="index").mean(axis=0)
    return annual_totals.loc[test_year - 1] * average_share


level_predictions = []
for test_year in (2024, 2025):
    history = seasonal_qc.loc[seasonal_qc["YR"] < test_year]
    target = seasonal_rf_xgb_predictions.loc[lambda rows: rows["year"].eq(test_year)]
    weekly_forecast = previous_total_by_week(history, test_year)
    threshold = history[seasonal_case].quantile(0.90)
    level_predictions.append(pd.DataFrame({
        "source_block": target.index,
        "year": test_year,
        "source_week": target["source_week"].to_numpy(),
        "actual_cases": target["actual_cases"].to_numpy(),
        "model": "Previous-year total",
        "forecast": target["source_week"].map(weekly_forecast).to_numpy(),
        "high_case_threshold": threshold,
    }))
seasonal_level_predictions = pd.concat(level_predictions, ignore_index=True)

level_errors = seasonal_level_predictions.drop(columns="high_case_threshold").copy()
level_errors["error"] = level_errors["forecast"] - level_errors["actual_cases"]
level_errors["absolute_error"] = level_errors["error"].abs()
level_errors["squared_error"] = level_errors["error"].pow(2)

level_comparison = pd.concat(
    [seasonal_errors.loc[seasonal_errors["year"].between(2024, 2025)], level_errors],
    ignore_index=True,
)
thresholds = seasonal_level_predictions.set_index("source_block")["high_case_threshold"]
level_comparison["high_case"] = level_comparison["actual_cases"].ge(
    level_comparison["source_block"].map(thresholds)
)
seasonal_level_yearly = summarize_seasonal_errors(level_comparison, ["year", "model"])
seasonal_level_combined = summarize_seasonal_errors(level_comparison, "model")
seasonal_level_high = summarize_seasonal_errors(
    level_comparison.loc[level_comparison["high_case"]], ["year", "model"]
)
display(seasonal_level_yearly.round(2))
display(seasonal_level_combined.round(2))
display(seasonal_level_high.round(2))
